<a href="https://colab.research.google.com/github/Delean-Mafra/faculdade/blob/main/Aplicacoes_da_Ciencia_de_Dadosem_Tecnologias_Disruptivas/GOTURN_(Generic_Object_Tracking_Using_Regression_Networks).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ============================================================
# Rastreamento com Ensemble + Kalman + Auto-Detecção (HOG)
# Cenário: loja de varejo movimentada (promoção)
# ============================================================

import os
import math
import base64
import urllib.request
import subprocess
import zipfile
from collections import deque

import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import HTML, display

# --- 1. DOWNLOAD E CONFIGURAÇÃO DO MODELO GOTURN ---
MODEL_DIR = '/content'
MODEL_TXT = f'{MODEL_DIR}/goturn.prototxt'
MODEL_BIN = f'{MODEL_DIR}/goturn.caffemodel'

def baixar_modelo_goturn():
    base_url = "https://raw.githubusercontent.com/opencv/opencv_extra/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking"
    if not os.path.exists(MODEL_TXT):
        urllib.request.urlretrieve(f"{base_url}/goturn.prototxt", MODEL_TXT)

    if not os.path.exists(MODEL_BIN) or os.path.getsize(MODEL_BIN) < 1000000:
        print("Montando pesos do modelo de Deep Learning...")
        partes = []
        for i in range(1, 5):
            nome_parte = f"goturn.caffemodel.zip.00{i}"
            caminho_parte = f"{MODEL_DIR}/{nome_parte}"
            partes.append(caminho_parte)
            if not os.path.exists(caminho_parte):
                urllib.request.urlretrieve(f"{base_url}/{nome_parte}", caminho_parte)

        zip_path = f"{MODEL_DIR}/goturn.caffemodel.zip"
        with open(zip_path, 'wb') as outfile:
            for parte in partes:
                with open(parte, 'rb') as infile:
                    outfile.write(infile.read())

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(MODEL_DIR)

baixar_modelo_goturn()

# --- 2. CONFIGURAÇÕES DE ENTRADA ---
VIDEO_IN = '/content/rastreamento_saida.mp4'
VIDEO_OUT = '/content/resultado_ensemble_final.mp4'
ZONA_PROMO_REL = (0.50, 0.30, 0.40, 0.60)
MAX_SALTO_REL = 0.20
MAX_FRAMES = 150

# --- 3. INICIALIZAÇÃO DE VÍDEO E AUTO-DETECÇÃO ---
if not os.path.exists(VIDEO_IN):
    raise FileNotFoundError(f'Vídeo não encontrado em: {VIDEO_IN}')

cap = cv2.VideoCapture(VIDEO_IN)
N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
ok_prev, f0 = cap.read()
if not ok_prev:
    raise RuntimeError('Não foi possível ler o primeiro frame.')

H0, W0 = f0.shape[:2]
ZONA_PROMO = (int(ZONA_PROMO_REL[0]*W0), int(ZONA_PROMO_REL[1]*H0),
              int(ZONA_PROMO_REL[2]*W0), int(ZONA_PROMO_REL[3]*H0))

# ---------------------------------------------------------
# NOVIDADE: Detecção Automática do Alvo usando HOG Descriptor
# ---------------------------------------------------------
print("Procurando pessoas no primeiro frame para inicializar os rastreadores...")
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# Detecta pessoas na imagem
caixas, pesos = hog.detectMultiScale(f0, winStride=(8, 8), padding=(8, 8), scale=1.05)

if len(caixas) > 0:
    # Se achou pessoas, pega a maior caixa (provavelmente a pessoa mais próxima da câmera)
    caixas_areas = [w * h for (x, y, w, h) in caixas]
    maior_caixa_idx = np.argmax(caixas_areas)
    x_hog, y_hog, w_hog, h_hog = caixas[maior_caixa_idx]

    # HOG costuma gerar caixas muito grandes, vamos ajustar para o torso/cabeça
    BBOX_INICIAL = (int(x_hog + w_hog*0.1), int(y_hog + h_hog*0.1), int(w_hog*0.8), int(h_hog*0.8))
    print(f"Alvo automático detectado na posição: {BBOX_INICIAL}")
else:
    # Fallback inspirado no seu arquivo teste.py (ROI centralizada)
    print("HOG não identificou pessoas. Usando ROI central baseada no tamanho do frame.")
    BBOX_INICIAL = (int(W0 * 0.4), int(H0 * 0.4), int(W0 * 0.2), int(H0 * 0.5))

# --- 4. FUNÇÕES CORE (ENSEMBLE + KALMAN) ---
def criar_trackers_ensemble():
    trackers = {}
    try: trackers['CSRT'] = cv2.TrackerCSRT_create()
    except: pass
    try:
        params = cv2.TrackerGOTURN_Params()
        params.modelTxt = MODEL_TXT
        params.modelBin = MODEL_BIN
        trackers['GOTURN'] = cv2.TrackerGOTURN_create(params)
    except: pass
    try:
        if hasattr(cv2, 'TrackerKCF_create'): trackers['KCF'] = cv2.TrackerKCF_create()
        elif hasattr(cv2.legacy, 'TrackerKCF_create'): trackers['KCF'] = cv2.legacy.TrackerKCF_create()
    except: pass
    return trackers

def criar_kalman(cx, cy):
    kf = cv2.KalmanFilter(4, 2)
    kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
    kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
    kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
    kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
    kf.errorCovPost = np.eye(4, dtype=np.float32)
    kf.statePost = np.array([[cx], [cy], [0], [0]], np.float32)
    return kf

def validar_bbox(b, W, H):
    x, y, w, h = [int(v) for v in b]
    return (max(0, min(x, W-1)), max(0, min(y, H-1)), max(1, min(w, W-x)), max(1, min(h, H-y)))

# --- 5. LOOP DE PROCESSAMENTO ---
writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W0, H0))
MAX_SALTO = MAX_SALTO_REL * W0
escala = max(W0 / 640.0, 0.6)
espessura = max(1, int(round(2 * escala)))

trackers = {}
kf = None
trilha = deque(maxlen=60)
stats = {'frames': 0, 'kalman_only': 0, 'na_zona': 0}
idx = 0

print("Iniciando o processamento do Ensemble...")
# Reinicia o vídeo para processar desde o frame zero
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:
    if MAX_FRAMES and idx >= MAX_FRAMES: break
    lido, frame = cap.read()
    if not lido: break

    if idx == 0:
        bbox_ini = validar_bbox(BBOX_INICIAL, W0, H0)
        trackers = criar_trackers_ensemble()
        for nome, t in trackers.items():
            t.init(frame, bbox_ini)

        x, y, w, h = bbox_ini
        cx, cy = x + w/2, y + h/2
        kf = criar_kalman(cx, cy)
        estado_str = f"INICIO ({len(trackers)} modelos)"
    else:
        pred = kf.predict()
        px, py = float(pred[0, 0]), float(pred[1, 0])

        caixas_validas = []
        for nome, t in trackers.items():
            ok, box = t.update(frame)
            if ok:
                bx, by, bw, bh = box
                tcx, tcy = bx + bw/2, by + bh/2
                if math.hypot(tcx - px, tcy - py) < MAX_SALTO:
                    caixas_validas.append((bx, by, bw, bh))

        if len(caixas_validas) > 0:
            caixas_np = np.array(caixas_validas)
            avg_box = np.mean(caixas_np, axis=0)
            avg_bx, avg_by, avg_bw, avg_bh = avg_box
            ccx, ccy = avg_bx + avg_bw/2, avg_by + avg_bh/2

            kf.correct(np.array([[ccx], [ccy]], np.float32))
            cx, cy, w, h = ccx, ccy, avg_bw, avg_bh
            estado_str = f"ENSEMBLE ({len(caixas_validas)}/{len(trackers)} ok)"
        else:
            kf.statePost = kf.statePre.copy()
            kf.errorCovPost = kf.errorCovPre.copy()
            cx, cy = px, py
            estado_str = "PREVISAO KALMAN (Oclusao)"
            stats['kalman_only'] += 1

    zx, zy, zw, zh = ZONA_PROMO
    na_zona = (zx <= cx <= zx+zw and zy <= cy <= zy+zh)
    stats['na_zona'] += int(na_zona)
    stats['frames'] += 1
    trilha.append((int(cx), int(cy)))

    # Desenho
    cv2.rectangle(frame, (zx, zy), (zx+zw, zy+zh), (255, 120, 0), espessura)
    cor_box = (0, 255, 0) if "ENSEMBLE" in estado_str else (0, 140, 255) if "KALMAN" in estado_str else (255, 255, 255)

    p1 = (int(cx - w/2), int(cy - h/2))
    p2 = (int(cx + w/2), int(cy + h/2))
    cv2.rectangle(frame, p1, p2, cor_box, espessura)

    for a, b in zip(trilha, list(trilha)[1:]):
        cv2.line(frame, a, b, (0, 255, 255), espessura)

    cv2.putText(frame, f'Status: {estado_str}', (10, int(30*escala)), cv2.FONT_HERSHEY_SIMPLEX, 0.7*escala, cor_box, espessura)
    writer.write(frame)
    idx += 1

cap.release()
writer.release()

# --- 6. EXIBIÇÃO DE RESULTADOS ---
print(f"Frames processados       : {stats['frames']}")
print(f"Tempo na zona promocional: {stats['na_zona'] / fps:.1f} s")

saida_h264 = '/content/resultado_ensemble_h264.mp4'
subprocess.run(['ffmpeg', '-y', '-i', VIDEO_OUT, '-vcodec', 'libx264', saida_h264, '-loglevel', 'quiet'], check=False)
if os.path.exists(saida_h264):
    with open(saida_h264, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

Procurando pessoas no primeiro frame para inicializar os rastreadores...
Alvo automático detectado na posição: (618, 310, 176, 353)
Iniciando o processamento do Ensemble...
Frames processados       : 150
Tempo na zona promocional: 0.3 s
